# Experiment: Run HPT with GridSearch to build a GBT model

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import string
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer 
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

import nltk
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize

from src import utils


# Parameters

In [3]:
RND_SEED = 123
PCT_TEST = 0.2
K_FOLD = 3

EXPERIMENT = "exp02_hpt_gbt"

# Paths
path_interim = os.path.join("data", "interim")
path_experiment =  os.path.join(path_interim, EXPERIMENT)

# Input
file_train = "train.csv"


# Output
file_exp = "df_exp_summary.csv"

In [4]:
utils.create_or_clean_folder(path_experiment)


Experiment folder already exists.
No files to clean, the folder is empty


# Load data

In [5]:
path_data_train = os.path.join(path_interim, file_train)

df_train = pd.read_csv(path_data_train)
df_train.head()

,x_text,y_is_nf
0,Poder crear cualquier mérito y asociarlo a mi ...,0
1,Como usuario autenticado quiero indicar que to...,0
2,Un usuario registrado o anónimo visualiza la l...,0
3,Como asesor quiero ver una lista de los experi...,0
4,Se podrá operar desde cualquier dispositivo a ...,1


# Build Pipeline

In [7]:
# Helper Cell: Tokenization and stemming in Spanish
import typing
import string


def tokenizer_stemmer_es(text) -> typing.List[str]:
    stopword_es = nltk.corpus.stopwords.words('spanish')
    stemmer = SnowballStemmer("spanish")

    clean_words = [word for word in word_tokenize(text) if word not in string.punctuation and word.lower() not in stopword_es] # list[str]
    return [stemmer.stem(word) for word in clean_words]  # list[str]


stopwords_es = nltk.corpus.stopwords.words('spanish')

example = df_train.loc[0, "x_text"]
ex_stem = tokenizer_stemmer_es(example)

print(f"{example=}")
print(f"{ex_stem=}")

example='Poder crear cualquier mérito y asociarlo a mi perfil de usuario.'
ex_stem=['pod', 'cre', 'cualqui', 'merit', 'asoci', 'perfil', 'usuari']


In [8]:
tfidf_unigrams = CountVectorizer(
    strip_accents="ascii",
    lowercase=True,
    tokenizer=tokenizer_stemmer_es,
    ngram_range=(1, 1),
    binary=True,
)


clf = GradientBoostingClassifier(
    n_estimators=2000,  # Many boosting rounds  so early stoping takes place
    validation_fraction=0.2,  # Early stopping
    random_state=RND_SEED)

# Create the pipeline
skl_pl = Pipeline([
    ('fte', tfidf_unigrams),
    ('clf', clf)
])


# GridSearch

GridSearchCV will run a set of Cross Validation experiments for you.
It will run for every combination of hiperparameters in the `param_grid`
and run a Cross Validation job for each.


Remember to use always the same number of CV Folds and the same CV metric on 
every experiment!


In [9]:
X_train = df_train['x_text']
y_train = df_train['y_is_nf']


param_grid = {
    'fte__max_features':[10, 50, 100, 150, 200],
    'fte__max_df': [0.1, 0.25, 0.5],
    'fte__min_df': [1, 3, 5],
    'clf__max_depth': [2, 5, 10, 20]
}



grid_search = GridSearchCV(
    skl_pl,
    param_grid,
    cv=K_FOLD,
    scoring='f1',
    n_jobs=-1
    )

# Fit GridSearchCV on the training data
grid_search.fit(X_train, y_train)
print(f"{grid_search.best_score_=}")
print(f"{grid_search.best_params_}")

/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/estebanm/Escritorio/Master/pc4_nlp/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/t

grid_search.best_score_=np.float64(0.6701219512195121)
{'clf__max_depth': 2, 'fte__max_df': 0.25, 'fte__max_features': 100, 'fte__min_df': 5}


In [10]:
df_exp_summary = pd.DataFrame(
    grid_search.cv_results_
)

df_exp_summary["experiment_id"] = EXPERIMENT
df_exp_summary.sort_values(ascending=True, by="rank_test_score").head(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf__max_depth,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
23,1.895222,0.158946,0.066013,0.004588,2,0.25,100,5,"{'clf__max_depth': 2, 'fte__max_df': 0.25, 'ft...",0.625000,0.800000,0.585366,0.670122,0.093252,1,exp02_hpt_gbt
38,2.184465,0.039063,0.068256,0.007357,2,0.50,100,5,"{'clf__max_depth': 2, 'fte__max_df': 0.5, 'fte...",0.625000,0.800000,0.585366,0.670122,0.093252,1,exp02_hpt_gbt
41,2.073582,0.070645,0.066723,0.004138,2,0.50,150,5,"{'clf__max_depth': 2, 'fte__max_df': 0.5, 'fte...",0.600000,0.716981,0.634146,0.650376,0.049117,3,exp02_hpt_gbt
26,2.044204,0.234791,0.069008,0.004804,2,0.25,150,5,"{'clf__max_depth': 2, 'fte__max_df': 0.25, 'ft...",0.600000,0.716981,0.634146,0.650376,0.049117,3,exp02_hpt_gbt
24,2.134710,0.029400,0.062239,0.000638,2,0.25,150,1,"{'clf__max_depth': 2, 'fte__max_df': 0.25, 'ft...",0.583333,0.705882,0.651163,0.646793,0.050126,5,exp02_hpt_gbt


# Diagnose the model

In [11]:
# Check DTM dimensions
skl_pl_fitted = grid_search.best_estimator_  

# Access the Vectorizer part of the pipeline
skl_pl_fte = skl_pl_fitted.named_steps['fte']

# Get DTM with transform()
dtm_train = skl_pl_fte.transform(X_train)
print(f"{dtm_train.shape=}")  # columns: Number of terms in the vocabulary

dtm_train.shape=(311, 100)


In [12]:
# Check training predictions and scoring

y_hats_train = skl_pl_fitted.predict(X_train)   # get preds with predict()
f1_score_train = f1_score(
    y_true=y_train,
    y_pred=y_hats_train
)

print(f"{f1_score_train=}")  # Is comparable to CV metric?

f1_score_train=1.0


# Write Experiments Results

In [13]:
df_exp_summary.to_csv(
    os.path.join(path_experiment, file_exp),
    index=False
)

# other experiments results and artifacts maybe useful

In [14]:
df_exp_summary

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf__max_depth,param_fte__max_df,param_fte__max_features,param_fte__min_df,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,experiment_id
0,1.668309,0.066840,0.067563,0.005173,2,0.1,10,1,"{'clf__max_depth': 2, 'fte__max_df': 0.1, 'fte...",0.200000,0.000000,0.153846,0.117949,0.085504,169,exp02_hpt_gbt
1,1.738559,0.129712,0.067385,0.004155,2,0.1,10,3,"{'clf__max_depth': 2, 'fte__max_df': 0.1, 'fte...",0.200000,0.000000,0.000000,0.066667,0.094281,173,exp02_hpt_gbt
2,1.761442,0.061884,0.068920,0.009369,2,0.1,10,5,"{'clf__max_depth': 2, 'fte__max_df': 0.1, 'fte...",0.200000,0.000000,0.153846,0.117949,0.085504,169,exp02_hpt_gbt
3,1.820123,0.216309,0.067368,0.002729,2,0.1,50,1,"{'clf__max_depth': 2, 'fte__max_df': 0.1, 'fte...",0.560000,0.486486,0.410256,0.485581,0.061136,144,exp02_hpt_gbt
4,1.725868,0.016893,0.070439,0.001321,2,0.1,50,3,"{'clf__max_depth': 2, 'fte__max_df': 0.1, 'fte...",0.358974,0.526316,0.410256,0.431849,0.070002,158,exp02_hpt_gbt
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,2.254409,0.065287,0.089216,0.002193,20,0.5,150,3,"{'clf__max_depth': 20, 'fte__max_df': 0.5, 'ft...",0.520000,0.680000,0.651163,0.617054,0.069630,23,exp02_hpt_gbt
176,2.270501,0.116381,0.068986,0.018924,20,0.5,150,5,"{'clf__max_depth': 20, 'fte__max_df': 0.5, 'ft...",0.520000,0.562500,0.604651,0.562384,0.034559,71,exp02_hpt_gbt
177,2.192467,0.078997,0.046834,0.001631,20,0.5,200,1,"{'clf__max_depth': 20, 'fte__max_df': 0.5, 'ft...",0.538462,0.633333,0.604651,0.592149,0.039727,43,exp02_hpt_gbt
178,1.963244,0.176398,0.039636,0.005600,20,0.5,200,3,"{'clf__max_depth': 20, 'fte__max_df': 0.5, 'ft...",0.509091,0.730769,0.636364,0.625408,0.090831,15,exp02_hpt_gbt
